# 9장 (실습 3): 클라우드 관리형 서비스의 지원 종료 버전 자동으로 찾기

> 책 『AI 보안, 클라우드에서 시작한다』 9장 [실습] 3번을 노트북에서 단계별로 실행하는 버전입니다. 관리형 DB, 쿠버네티스, 서버리스, 캐시의 버전 조회 결과를 공개된 지원 종료 일정과 자동으로 대조해, 이미 지원이 끝난 것과 곧 끝나는 것, 자동 마이너 업그레이드가 꺼진 것을 찾습니다. 소요 시간은 30분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. endoflife.date 공개 API에서 서비스·버전별 **지원 종료일**을 받아 읽는 법을 익힙니다.
2. 관리형 서비스의 조회 결과를 지원 종료 일정과 **자동으로 대조**해 긴급·위험·주의를 판정합니다.
3. 기준일을 바꿔 보며 **오늘 양호한 리소스도 곧 위험이 된다**는 것을 확인하고, 이 대조를 정기 작업으로 만드는 방법을 정리합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | 예정 |
| 1 | 지원 종료 일정을 API로 확인해 보기 | 예정 |
| 2 | 자동 대조 실행 | 예정 |
| 3 | 결과 읽기 | 예정 |
| 4 | 매주 자동으로 돌게 만들기 | 예정 |
| 5 | (선택) 내 AWS 계정으로 해 보기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

## 실습 시나리오

9-1과 9-3에서 본 것처럼 관리형 서비스도 버전이 있고, 그 버전의 지원 종료는 고객이 챙겨야 합니다. 공급자는 지원이 끝난 버전을 강제로 올리거나(서비스 중단 위험), 연장 지원 비용을 청구하거나, 보안 패치를 더는 내지 않습니다. 서비스마다 지원 일정 문서가 따로 있어서 사람이 하나씩 찾아 대조하면 금방 밀립니다. 9-4의 자동 패치 순환처럼 **"현황 수집 → 외부 정보와 대조 → 조치 대상 생성"** 을 스크립트로 만들어 봅니다.

```text
 sample/*.json (aws ... describe 결과) ─┐
                                       ├─▶ eol_check.py ─▶ 긴급 / 위험 / 주의 / 확인 필요 / 양호 ─▶ 종료 코드 1이면 조치
 endoflife.date API (버전별 지원 종료일) ┘
```

> 🔎 (용어 풀이)
> - **메이저·마이너 버전**: `14.15`에서 앞의 `14`가 메이저, 뒤의 `15`가 마이너 버전입니다. 마이너 업그레이드는 버그·보안 수정이 중심이라 대부분 그대로 호환되고, 메이저 업그레이드는 기능이 바뀌어 호환성 시험이 필요합니다.
> - **자동 마이너 업그레이드**: 관리형 DB 같은 서비스에서 켜 두면, 공급자가 정해진 점검 시간에 마이너 버전을 알아서 올려 주는 설정입니다.
> - **지원 종료(EOL, End of Life)**: 공급자가 그 버전에 보안 패치를 더 내지 않는 시점입니다. 9장 본문의 EOS(End of Support)와 같은 뜻으로 씁니다.

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Python 3만 있으면 되고 외부 패키지가 필요 없습니다. 예시 조회 결과로 진행하므로 클라우드 계정과 API Key가 필요 없습니다. 지원 종료 일정은 endoflife.date의 공개 API에서 받으므로 인터넷 연결이 필요합니다.
>
> **2. 노트북 열기**<br>
> 처음이라면 [SETUP.md](../SETUP.md)의 6단계(VS Code와 Jupyter 노트북 연결)를 먼저 한 번 해 둡니다. 그다음에는 아래 두 방법 가운데 하나로 이 노트북을 엽니다.
> - **VS Code**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs && code .`를 실행합니다. 창 왼쪽 아래에 `WSL: Ubuntu-24.04`가 보이면, 왼쪽 탐색기에서 `ch09-ops-tracing-sw-inventory` 폴더를 펼쳐 이 노트북을 엽니다. 그리고 오른쪽 위 커널 표시를 눌러 **Select Another Kernel...** → **Python Environments...** → `.venv-lab`을 고릅니다. 목록에 `.venv-lab`이 없으면 6단계 ⑥의 2번처럼 `Python: Select Interpreter` → **Enter interpreter path...**로 `~/.venv-lab/bin/python`의 위치를 먼저 알려 줍니다.
> - **웹 브라우저(Jupyter Lab)**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs/ch09-ops-tracing-sw-inventory && ~/.venv-lab/bin/jupyter lab`을 실행하고, 화면에 나온 `http://localhost:8888/...` 주소를 Windows의 브라우저에서 엽니다.
> - 노트북 파일만 따로 내려받거나 다른 폴더로 옮기지 마세요. 노트북은 자기가 들어 있는 `ch09-ops-tracing-sw-inventory` 폴더의 실습 파일을 읽습니다. 위 두 방법처럼 저장소 안에서 열면 따로 신경 쓸 것은 없습니다.
>
> **3. 기준 버전**<br>
> - 이 노트북은 2026년 10월 23일 Ubuntu 24.04(WSL2), Python 3.12에서 끝까지 실행해 확인했습니다. 실행 결과가 저장된 버전은 `ch09_lab3_cloud_eol_check_executed.ipynb`입니다.

> 📌 (참고) 판정은 **실행한 날**을 기준으로 합니다. 책과 README의 표(2026년 10월 20일 기준)와 남은 일수가 다르고, 그 사이 endoflife.date의 일정이 바뀌었다면 판정도 다를 수 있습니다.

## 0. 실습 준비와 실습 파일

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 지원 종료 일정을 API로 확인해 보기 | 예정 |
| 2 | 자동 대조 실행 | 예정 |
| 3 | 결과 읽기 | 예정 |
| 4 | 매주 자동으로 돌게 만들기 | 예정 |
| 5 | (선택) 내 AWS 계정으로 해 보기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "python3": "python3 --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

| 파일 | 역할 |
|---|---|
| `cloud-eol/sample/` | 가상 회사의 AWS 조회 결과 예시. AWS CLI가 내는 JSON 형식 그대로이며, 필요한 필드만 남겼음 |
| `cloud-eol/eol_check.py` | 조회 결과를 읽어 서비스·버전별 지원 종료일을 endoflife.date에서 받아 대조하고 판정. 긴급·위험이 있으면 종료 코드 1 |
| `cloud-eol/collect_aws.sh` | (선택) 자기 AWS 계정에서 같은 형식의 조회 결과를 만드는 스크립트. 조회(describe, list)만 함 |

실습 파일을 `~/eol-lab`으로 복사하고, 예시 조회 결과 하나를 열어 봅니다.

In [ ]:
import shutil
import sys

WORK = Path.home() / "eol-lab"
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(LAB / "cloud-eol", WORK)
print((WORK / "sample" / "rds.json").read_text())

| 파일 | 만든 명령 | 판정에 쓰는 값 |
|---|---|---|
| `rds.json` | `aws rds describe-db-instances` | 엔진, 엔진 버전, 자동 마이너 업그레이드 |
| `elasticache.json` | `aws elasticache describe-cache-clusters` | 엔진, 엔진 버전, 자동 마이너 업그레이드 |
| `eks.json` | `aws eks describe-cluster --name <클러스터>` (클러스터마다) | 클러스터 버전 |
| `lambda.json` | `aws lambda list-functions` | 함수별 런타임 |

`eol_check.py`의 `resources()`로 네 파일에서 리소스를 꺼내 봅니다.

In [ ]:
sys.path.insert(0, str(WORK))
import eol_check as ec

RES = list(ec.resources(str(WORK / "sample")))
table(["리소스", "서비스", "엔진", "버전", "자동 마이너 업그레이드"],
      [(n, s, e, v, "-" if a is None else ("켜짐" if a else "꺼짐")) for n, s, e, v, a in RES])

## 1. 지원 종료 일정을 API로 확인해 보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 지원 종료 일정을 API로 확인해 보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 자동 대조 실행 | 예정 |
| 3 | 결과 읽기 | 예정 |
| 4 | 매주 자동으로 돌게 만들기 | 예정 |
| 5 | (선택) 내 AWS 계정으로 해 보기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

In [ ]:
%%bash
curl -s https://endoflife.date/api/amazon-eks.json | python3 -m json.tool | head -9

endoflife.date는 수백 개 제품의 버전별 출시일과 지원 종료일을 모아 공개하는 오픈소스 프로젝트이고, 공급자의 공식 문서를 출처로 합니다. 버전 구간(`cycle`)마다 표준 지원 종료일(`eol`)과 연장 지원 종료일(`extendedSupport`)이 있습니다. EKS의 최근 버전 구간을 표로 봅니다.

In [ ]:
eks = ec.cycles("amazon-eks")
table(["버전(cycle)", "출시일", "표준 지원 종료(eol)", "연장 지원 종료"],
      [(c, d.get("releaseDate"), d.get("eol"), d.get("extendedSupport")) for c, d in list(eks.items())[:8]])

서비스에 따라 필드의 뜻이 조금 다르므로 `eol_check.py`는 다음처럼 읽습니다.

`[표 실습9-2]` 서비스별 지원 종료일을 읽는 방법

| 서비스 (API 제품 이름) | 보안 패치가 끝나는 날로 쓰는 값 | 그 뒤 |
|---|---|---|
| EKS (`amazon-eks`) | `eol`: 표준 지원 종료 | `extendedSupport`까지 추가 비용을 내고 연장 지원 |
| RDS PostgreSQL (`amazon-rds-postgresql`) | `eol`: 메이저 버전 표준 지원 종료 | `extendedSupport`까지 추가 비용을 내고 연장 지원 |
| RDS MySQL (`amazon-rds-mysql`) | `support`: 이 마이너 버전의 지원 종료 (메이저 버전보다 먼저 끝남) | 마이너 업그레이드로 해결 |
| Lambda (`aws-lambda`) | `support`: 런타임 지원 중단, 보안 패치 끝 | `eol`부터 함수 수정까지 막힘 |
| ElastiCache Redis (`amazon-elasticache-redis`) | `eol`: 표준 지원 종료 | `extendedSupport`까지 연장 지원 |

> 📌 (참고) endoflife.date는 커뮤니티가 관리하는 정보입니다. 판정 결과로 실제 조치를 정하기 전에는 공급자의 공식 문서("Amazon EKS Kubernetes release calendar", "Lambda runtimes deprecation" 등)와 AWS Health 대시보드의 알림으로 한 번 더 확인합니다.

`find_cycle()`은 자세한 버전부터 찾고, 없으면 한 단계씩 줄여 갑니다. 예시 리소스의 버전이 어느 구간에 맞춰졌는지 봅니다.

In [ ]:
rows = []
for n, s, e, v, a in RES:
    if v.startswith("("):
        continue
    c = ec.find_cycle(ec.PRODUCT[e], v)
    rows.append((n, ec.PRODUCT[e], v, c["cycle"] if c else "(없음)", c.get("support") if c else "-", c.get("eol") if c else "-"))
table(["리소스", "API 제품", "버전", "맞춘 구간", "support", "eol"], rows)

## 2. 자동 대조 실행

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 지원 종료 일정을 API로 확인해 보기 | ✅ 완료 |
| 2 | 자동 대조 실행 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 결과 읽기 | 예정 |
| 4 | 매주 자동으로 돌게 만들기 | 예정 |
| 5 | (선택) 내 AWS 계정으로 해 보기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

In [ ]:
%%bash
cd ~/eol-lab
python3 eol_check.py sample; echo "종료 코드: $?"

`[표 실습9-3]` 예시 회사의 판정 결과 — 위 출력이 이번 실행의 결과입니다. 긴급·위험이 하나라도 있으면 종료 코드 1이 나옵니다.

## 3. 결과 읽기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 지원 종료 일정을 API로 확인해 보기 | ✅ 완료 |
| 2 | 자동 대조 실행 | ✅ 완료 |
| 3 | 결과 읽기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 매주 자동으로 돌게 만들기 | 예정 |
| 5 | (선택) 내 AWS 계정으로 해 보기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

- **"관리형"인데 지원이 이미 끝난 것이 있습니다.** order-db(PostgreSQL 13), prod-cluster(EKS 1.33), resize-image(python3.9)는 표준 지원이 끝났습니다. RDS와 EKS는 서비스가 멈추지는 않지만, 그 대신 연장 지원 비용이 자동으로 청구되고 있을 수 있습니다. 비용 청구서에서 "Extended Support" 항목을 찾아보면 지원 종료를 놓친 리소스가 드러나기도 합니다.
- **Lambda 런타임은 지원이 끝나도 함수가 계속 돕니다.** 아무도 알아채지 못하는 사이 패치 없는 런타임이 계속 운영되는 구조입니다. 마지막으로 고친 지 오래된 함수(`LastModified`)일수록 이런 경우가 많습니다(아래 셀).
- **같은 "8.4"라도 마이너 버전마다 지원 종료일이 다릅니다.** billing-db는 메이저 버전 8.4보다 마이너 버전 8.4.7의 지원이 먼저 끝납니다. 자동 마이너 업그레이드가 켜져 있었다면 저절로 해결되었을 일입니다.
- **컨테이너 이미지로 배포한 함수는 이 방법으로 볼 수 없습니다.** rag-indexer처럼 이미지로 배포한 Lambda는 런타임 정보가 없습니다. 이미지 안의 OS와 런타임은 8장 실습 1처럼 이미지를 스캔해야 보입니다.
- **EKS는 클러스터 버전만 본 것입니다.** 공급자가 컨트롤 플레인을 관리하더라도, 업그레이드 시점과 워커 노드, 애드온은 고객 몫입니다(표 9-5).

> 🔎 (용어 풀이) 쿠버네티스 클러스터는 두 부분으로 나뉩니다. **컨트롤 플레인**은 클러스터 전체를 관리하는 두뇌(API Server 등)로 EKS에서는 AWS가 운영하고, **워커 노드**는 실제 컨테이너가 도는 서버로 고객이 관리합니다. **애드온**은 네트워크, DNS 같은 클러스터의 부가 기능입니다. 컨트롤 플레인 버전을 올려도 워커 노드와 애드온은 따로 올려야 합니다.

In [ ]:
from datetime import date

funcs = json.loads((WORK / "sample" / "lambda.json").read_text())["Functions"]
table(["함수", "런타임", "마지막 수정", "수정한 지 며칠"],
      [(f["FunctionName"], f.get("Runtime") or "(이미지)", f["LastModified"][:10],
        (date.today() - date.fromisoformat(f["LastModified"][:10])).days) for f in funcs])

### 3.1. 기준일을 바꿔 보기

판정 기준(180일)은 `--days`로, 기준일은 `--today`로 바꿀 수 있습니다. 오늘부터 6개월 간격으로 기준일을 옮기며 판정이 어떻게 바뀌는지 셉니다. 메이저 업그레이드는 호환성 시험까지 몇 달이 걸리므로, 6개월 전에는 계획이 시작되어야 합니다.

In [ ]:
from collections import Counter

VERDICTS = ["긴급", "위험", "주의", "확인 필요", "양호"]


def verdicts(today):
    out = subprocess.run([sys.executable, "eol_check.py", "sample", "--today", str(today)], cwd=WORK,
                         capture_output=True, text=True).stdout
    return Counter(v for line in out.splitlines()[2:] for v in VERDICTS if f"  {v}  " in line)


t0 = date.today()
rows = []
for months in [0, 6, 12, 18]:
    d = date(t0.year + (t0.month - 1 + months) // 12, (t0.month - 1 + months) % 12 + 1, min(t0.day, 28))
    c = verdicts(d)
    rows.append((f"{d} (+{months}개월)", *(c[v] for v in VERDICTS)))
table(["기준일", *VERDICTS], rows)

아무것도 바꾸지 않아도 시간이 지나면 "양호"가 "위험"으로, "위험"이 "긴급"으로 옮겨 갑니다. 지원 종료 대조는 한 번 하고 끝나는 점검이 아니라 **정기적으로 돌려야 하는 작업**입니다.

## 4. 매주 자동으로 돌게 만들기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 지원 종료 일정을 API로 확인해 보기 | ✅ 완료 |
| 2 | 자동 대조 실행 | ✅ 완료 |
| 3 | 결과 읽기 | ✅ 완료 |
| 4 | 매주 자동으로 돌게 만들기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | (선택) 내 AWS 계정으로 해 보기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

`eol_check.py`는 긴급·위험이 있으면 종료 코드 1을 돌려줍니다. 실습 1처럼 예약 작업으로 등록하면, 사람이 기억하지 않아도 매주 대조가 돌고 결과가 패치 큐로 넘어갑니다.

```bash
# 매주 월요일 08:00: 조회 → 대조 → 조치가 필요하면 결과를 패치 큐(예: 티켓 시스템 수신 메일)로 보냄
0 8 * * 1 cd /opt/cloud-eol && ./collect_aws.sh weekly && python3 eol_check.py weekly > eol.txt || mail -s "지원 종료 점검: 조치 필요" patch-queue@example.com < eol.txt
```

이 흐름이 9-4 자동 패치 순환의 ①(외부 정보 수집)·②(자산 대조)·③(조치 작업 생성) 단계입니다. 실무에서는 결과를 티켓 시스템 API로 바로 등록하고, 리소스의 담당 팀 태그를 읽어 담당자를 자동으로 지정합니다.

## 5. (선택) 내 AWS 계정으로 해 보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 지원 종료 일정을 API로 확인해 보기 | ✅ 완료 |
| 2 | 자동 대조 실행 | ✅ 완료 |
| 3 | 결과 읽기 | ✅ 완료 |
| 4 | 매주 자동으로 돌게 만들기 | ✅ 완료 |
| 5 | (선택) 내 AWS 계정으로 해 보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

AWS 계정이 있다면 **읽기 전용 권한**(예: AWS 관리형 정책 `ReadOnlyAccess`)으로 조회 결과를 만들어 같은 스크립트에 넣습니다. 조직 계정이라면 보안팀에 읽기 전용 역할을 요청합니다. `collect_aws.sh`는 조회(describe, list)만 합니다.

In [ ]:
print((WORK / "collect_aws.sh").read_text())

아래 셀은 기본값으로 실행하지 않습니다. AWS CLI와 읽기 전용 자격 증명이 준비되었다면 `RUN_AWS = True`와 리전을 고친 뒤 실행합니다.

> 📌 (참고) 조회는 **지정한 리전**만 합니다. 잊힌 리전에 오래된 리소스가 남아 있는 경우가 많으니, 쓰는 리전을 모두 확인합니다. 자기 계정에서는 결과를 콘솔과 한 번 대조해 봅니다.

In [ ]:
RUN_AWS = False
REGION = "ap-northeast-2"

if RUN_AWS:
    print(sh(f"cd ~/eol-lab && chmod +x collect_aws.sh && ./collect_aws.sh my-account {REGION} 2>&1"))
    print(sh("cd ~/eol-lab && python3 eol_check.py my-account"))
else:
    print("RUN_AWS = False: 건너뜀")

Azure와 GCP도 같은 방식으로 할 수 있습니다. 아래 명령으로 버전을 조회하고, endoflife.date의 `azure-kubernetes-service`, `google-kubernetes-engine` 같은 제품으로 대조합니다.

```bash
# Azure: App Service·Functions 런타임, AKS 버전
az webapp list --query "[].{name:name, runtime:siteConfig.linuxFxVersion}" -o table
az aks list --query "[].{name:name, version:kubernetesVersion}" -o table

# GCP: Cloud Functions 런타임, GKE 버전, Cloud SQL 버전
gcloud functions list --format="table(name,runtime)"
gcloud container clusters list --format="table(name,currentMasterVersion)"
gcloud sql instances list --format="table(name,databaseVersion)"
```

## 6. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 지원 종료 일정을 API로 확인해 보기 | ✅ 완료 |
| 2 | 자동 대조 실행 | ✅ 완료 |
| 3 | 결과 읽기 | ✅ 완료 |
| 4 | 매주 자동으로 돌게 만들기 | ✅ 완료 |
| 5 | (선택) 내 AWS 계정으로 해 보기 | ✅ 완료 |
| 6 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 정리 | 예정 |

### 6.1. 체크포인트

- [ ] "관리형 서비스니까 공급자가 알아서 패치한다"고 생각했던 부분 중, 실제로는 고객 몫인 것은 무엇이었나요?
- [ ] 지원이 끝났는데도 서비스가 계속 돌아가는 경우(연장 지원, Lambda 런타임)는 왜 더 위험할까요?
- [ ] 자동 마이너 업그레이드를 켜 두면 해결되는 항목과, 사람이 계획해야 하는 항목(메이저 업그레이드)을 구분할 수 있나요?
- [ ] 이 대조를 매주 자동으로 돌리려면, 실습 1의 서버 현황 수집과 무엇을 합쳐야 회사 전체를 볼 수 있을까요?

### 6.2. 과제 1: 우리 회사 리소스 추가해 보기

`~/eol-lab/mine` 폴더에 예시를 복사한 뒤, 리소스를 더해 판정을 확인합니다. AWS CLI가 내는 형식과 같아야 `eol_check.py`가 읽을 수 있습니다.

1. 아래 셀의 `NEW_DB`, `NEW_FUNC`를 원하는 엔진·버전·런타임으로 바꿔 봅니다. (예: MySQL `8.0.40`, PostgreSQL `14.15`, Lambda `nodejs18.x`, `java11`)
2. 판정과 비고를 읽고, 각 리소스를 **누가 언제까지** 무엇을 해야 하는지 한 줄씩 적어 봅니다.

In [ ]:
MINE = WORK / "mine"
shutil.rmtree(MINE, ignore_errors=True)
shutil.copytree(WORK / "sample", MINE)

# TODO: 엔진·버전·런타임을 바꿔 봅니다
NEW_DB = {"DBInstanceIdentifier": "legacy-erp-db", "Engine": "mysql", "EngineVersion": "8.0.40",
          "AutoMinorVersionUpgrade": False}
NEW_FUNC = {"FunctionName": "old-webhook", "Runtime": "nodejs18.x", "LastModified": "2024-03-02T10:00:00.000+0000"}

rds = json.loads((MINE / "rds.json").read_text())
rds["DBInstances"].append(NEW_DB)
(MINE / "rds.json").write_text(json.dumps(rds, indent=1))
lam = json.loads((MINE / "lambda.json").read_text())
lam["Functions"].append(NEW_FUNC)
(MINE / "lambda.json").write_text(json.dumps(lam, indent=1))

out = subprocess.run([sys.executable, "eol_check.py", "mine"], cwd=WORK, capture_output=True, text=True).stdout
print("\n".join(l for l in out.splitlines() if "legacy-erp-db" in l or "old-webhook" in l or l.startswith(("기준일", "총"))))

> 📌 (참고) 버전을 endoflife.date에 없는 값으로 쓰면 "확인 필요: 지원 일정을 찾지 못함"이 나옵니다. 자동 대조에서 **찾지 못한 것을 양호로 처리하지 않는** 것이 중요합니다.

### 6.3. 과제 2: 조치 티켓 설계 (서술)

판정 결과를 티켓으로 바꾼다고 할 때, 아래 질문에 답해 보세요.

1. "긴급"과 "위험"의 처리 기한을 각각 어떻게 정할까요? 연장 지원 비용이 청구되는 리소스는 기한을 다르게 둘까요?
2. 리소스마다 담당 팀을 자동으로 정하려면 조회 결과에 무엇이 더 있어야 할까요? (태그, 계정 구조)

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

1.
2.

## 7. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 지원 종료 일정을 API로 확인해 보기 | ✅ 완료 |
| 2 | 자동 대조 실행 | ✅ 완료 |
| 3 | 결과 읽기 | ✅ 완료 |
| 4 | 매주 자동으로 돌게 만들기 | ✅ 완료 |
| 5 | (선택) 내 AWS 계정으로 해 보기 | ✅ 완료 |
| 6 | 체크포인트와 과제 | ✅ 완료 |
| 7 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [ ]:
sys.modules.pop("eol_check", None)
shutil.rmtree(WORK, ignore_errors=True)
print("실습 폴더 삭제됨")

---
> 다음 실습 4에서는 찾은 조치 대상을 실제로 패치하고, 결과를 다시 점검해 확인하고, 패치 전에는 Workaround로 막는 과정을 직접 돌려 봅니다.